# Checking each cleaning function (Peter-work)

`clean_peter.py` holds the functions; this notebook calls them one at a time and checks the result.
Rules and decisions (D1 to D7) come from `plan.md`.

Each section: **what the function should do**, then a run on the real data, then `assert` checks.
If an assert fails the cell stops with an error, so a clean run of the whole notebook means every check passed.

Run from the project folder with the raw files in `data/raw/`.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import clean_peter as cp

pd.set_option("display.max_colwidth", 80)
prices_raw, news_raw, filings_raw = cp.load_raw()
pd.DataFrame({"rows": [len(prices_raw), len(news_raw), len(filings_raw)],
              "cols": [prices_raw.shape[1], news_raw.shape[1], filings_raw.shape[1]]},
             index=["prices", "news", "filings"])

,rows,cols
prices,2656,9
news,3888,4
filings,73,6


## 1. `clean_common`
Shared rules: `id` to int64, `symbol` stripped and uppercased as category, `date` to datetime64, sorted by symbol and date. Must never drop rows.

First a tiny hand-made table with messy values, so we can see exactly what changes.

In [2]:
toy = pd.DataFrame({
    "id": [3, 1, 2],
    "symbol": [" msft", "AAPL ", "aapl"],
    "date": ["2025-01-03", "2025-01-02", "2025-01-01"],
})
toy_clean = cp.clean_common(toy)
toy_clean

,id,symbol,date
0,2,AAPL,2025-01-01
1,1,AAPL,2025-01-02
2,3,MSFT,2025-01-03


In [3]:
assert list(toy_clean["symbol"]) == ["AAPL", "AAPL", "MSFT"]
assert toy_clean["date"].is_monotonic_increasing or toy_clean.groupby("symbol", observed=True)["date"].apply(lambda s: s.is_monotonic_increasing).all()
assert str(toy_clean["date"].dtype).startswith("datetime64")
assert len(toy_clean) == len(toy)
assert len(toy) == 3 and toy["symbol"].iloc[0] == " msft"   # original table untouched (.copy())
print("clean_common: all checks passed")

clean_common: all checks passed


## 2. `latest_date_mentioned`
Finds the latest explicit date written in a text, in `2025-02-25` or `February 25, 2025` form. Used for the look-ahead flag (refinement 2).

In [4]:
cases = {
    "Closed at 239.58 on 2024-12-02.": "2024-12-02",
    "On February 25, Adobe launched ... reported Feb 25, 2025 and 2025-02-14.": "2025-02-25",
    "No explicit date here.": None,
    "Avoids facts dated after 2025-10-14.": "2025-10-14",
}
for text, expected in cases.items():
    got = cp.latest_date_mentioned(text)
    print(f"{str(got.date()) if pd.notna(got) else None!s:12}  <- {text}")
    assert (pd.isna(got) and expected is None) or str(got.date()) == expected
print("latest_date_mentioned: all checks passed")

2024-12-02    <- Closed at 239.58 on 2024-12-02.
2025-02-25    <- On February 25, Adobe launched ... reported Feb 25, 2025 and 2025-02-14.
None          <- No explicit date here.
2025-10-14    <- Avoids facts dated after 2025-10-14.
latest_date_mentioned: all checks passed


Note the last case: a disclaimer like "avoids facts dated after 2025-10-14" only returns the row's own date, so it is **not** flagged, because the flag needs a date later than the row date. That is exactly the false positive refinement 2 removed.

## 3. `clean_prices`
Adds `daily_return` from `adj_close` within each symbol (D6) and `flag_large_move` for moves above 12% (D5). No rows dropped.

In [5]:
prices = cp.clean_prices(prices_raw)
print(prices.shape)
prices.head(3)

(2656, 11)


,id,symbol,date,open,high,low,close,adj_close,volume,daily_return,flag_large_move
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100,NaN,False
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000,0.012772,False
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900,0.001484,False


In [6]:
assert len(prices) == len(prices_raw)
assert prices["daily_return"].isna().sum() == prices["symbol"].nunique()   # only first day per symbol
assert cp.ohlc_violations(prices) == 0

# recompute one return by hand
aapl = prices[prices["symbol"] == "AAPL"]
manual = aapl["adj_close"].iloc[1] / aapl["adj_close"].iloc[0] - 1
assert abs(manual - aapl["daily_return"].iloc[1]) < 1e-12
print("clean_prices: all checks passed")

clean_prices: all checks passed


In [7]:
prices.loc[prices["flag_large_move"], ["symbol", "date", "close", "daily_return"]]

,symbol,date,close,daily_return
87,AAPL,2025-04-09,198.850006,0.153288
340,ADBE,2024-12-12,474.630005,-0.136926
400,ADBE,2025-03-13,377.839996,-0.138532
1415,META,2025-04-09,585.770020,0.147556
2028,NVDA,2025-01-27,118.419998,-0.169682
2079,NVDA,2025-04-09,114.330002,0.187227
2389,TSLA,2025-03-10,222.149994,-0.154262
2411,TSLA,2025-04-09,272.200012,0.226900
2450,TSLA,2025-06-05,284.700012,-0.142599


All 9 match known market events (2025-04-09 tariff pause rally, NVDA DeepSeek drop on 2025-01-27), so they stay in the data with a flag.

## 4. `trading_calendar`
List of trading dates from the cleaned prices, used to line up weekend news with the next session.

In [8]:
days = cp.trading_calendar(prices)
assert days.is_monotonic_increasing and days.is_unique
assert (days.dt.dayofweek < 5).all()                         # no weekends
assert len(days) == prices.groupby("symbol", observed=True).size().iloc[0]
print(len(days), "trading days from", days.min().date(), "to", days.max().date())

332 trading days from 2024-12-02 to 2026-03-31


## 5. `clean_news`
Wording is never edited (D3). Adds `text_len`, `flag_placeholder` (D1, refinement 1), `latest_date_mentioned` and `flag_lookahead` (D3, refinement 2), `is_trading_day` and `next_trading_date` (D2).

In [9]:
news = cp.clean_news(news_raw, days)
print(news.shape)
news.drop(columns="highlights").head(3)

(3888, 10)


,id,symbol,date,text_len,flag_placeholder,latest_date_mentioned,flag_lookahead,is_trading_day,next_trading_date
0,1,AAPL,2024-12-01,1586,False,2024-12-01,False,False,2024-12-02
1,2,AAPL,2024-12-02,2077,False,2024-12-02,False,True,2024-12-02
2,3,AAPL,2024-12-03,1081,False,NaT,False,True,2024-12-03


In [10]:
assert len(news) == len(news_raw)
raw_text = news_raw.sort_values(["symbol", "date"])["highlights"].str.strip().reset_index(drop=True)
assert raw_text.equals(news["highlights"])                       # wording unchanged
assert news["next_trading_date"].notna().all()
assert (news["next_trading_date"] >= news["date"]).all()
assert news["next_trading_date"].isin(days).all()
assert (news.loc[news["is_trading_day"], "next_trading_date"] == news.loc[news["is_trading_day"], "date"]).all()
print("clean_news: all checks passed")

clean_news: all checks passed


**Placeholder rows.** Read them: each should be an empty or broken summary, not real news.

In [11]:
news.loc[news["flag_placeholder"], ["symbol", "date", "text_len", "highlights"]]

,symbol,date,text_len,highlights
542,ADBE,2025-01-26,25,The article also includes
1004,AMZN,2025-01-02,153,"This item conveys a clear negative signal for that specific date, indicating..."
1468,GOOGL,2024-12-11,352,I can't provide any historical news or updates regarding GOOGL Google Alphab...
1486,GOOGL,2024-12-29,397,"There are no substantive GOOGL news items, events, filings, earnings, or mar..."
1488,GOOGL,2024-12-31,138,"Consequently, there are no substantive GOOGL events, themes, or market senti..."
1957,META,2024-12-14,222,"The sole\n\nAs a result, no themes, developments, or market sentiment relate..."
1961,META,2024-12-18,288,"Because no META content is present in the supplied material for that date, t..."
2449,MSFT,2024-12-20,374,"Because no substantive MSFT events, earnings, product launches, analyst note..."


**Look-ahead rows.** Text mentions a date later than its own row date.

In [12]:
leak = news.loc[news["flag_lookahead"]]
for _, row in leak.iterrows():
    t = row["highlights"]
    i = t.find("February 25")
    print(row["symbol"], row["date"].date(), "mentions", row["latest_date_mentioned"].date())
    print("...", t[max(0, i - 150): i + 120].replace("\n", " "), "...")

ADBE 2025-02-15 mentions 2025-02-25
...  February 15, Adobe did not post content on its official social channels and that its social activity during that window was limited. In contrast, on February 25, Adobe announced the launch of Photoshop for iPhone, extending its image editing and design capabilities to  ...


**Weekend news** keeps its own date and gets the next trading date for joins with prices.

In [13]:
news.loc[~news["is_trading_day"], ["symbol", "date", "next_trading_date"]].head(4)

,symbol,date,next_trading_date
0,AAPL,2024-12-01,2024-12-02
6,AAPL,2024-12-07,2024-12-09
7,AAPL,2024-12-08,2024-12-09
13,AAPL,2024-12-14,2024-12-16


## 6. `clean_filings`
Strips whitespace in the text only. Adds `mda_len`, `risk_len`, `flag_risk_repeat` and `flag_risk_reference_only` (D4).

In [14]:
filings = cp.clean_filings(filings_raw)
print(filings.shape)
filings.drop(columns=["mda_content", "risk_content"]).head(3)

(73, 10)


,id,symbol,date,document_type,mda_len,risk_len,flag_risk_repeat,flag_risk_reference_only
0,1,AAPL,2024-02-01,10-Q,15926,4588,False,False
1,2,AAPL,2024-05-02,10-Q,17486,4800,False,False
2,3,AAPL,2024-08-01,10-Q,18121,982,False,True


In [15]:
assert len(filings) == len(filings_raw)
assert set(filings["document_type"].unique()) <= {"10-K", "10-Q"}
assert not filings["mda_content"].str.startswith(" ").any()
assert not filings.groupby("symbol", observed=True)["flag_risk_repeat"].first().any()  # first filing can't repeat
print("clean_filings: all checks passed")

clean_filings: all checks passed


In [16]:
filings.loc[filings["flag_risk_repeat"] | filings["flag_risk_reference_only"],
            ["symbol", "date", "document_type", "risk_len", "flag_risk_repeat", "flag_risk_reference_only"]]

,symbol,date,document_type,risk_len,flag_risk_repeat,flag_risk_reference_only
2,AAPL,2024-08-01,10-Q,982,False,True
8,AAPL,2026-01-30,10-Q,436,False,True
22,AMZN,2024-10-31,10-Q,59935,True,False
25,AMZN,2025-07-31,10-Q,60653,True,False
26,AMZN,2025-10-30,10-Q,60653,True,False
33,GOOGL,2025-04-24,10-Q,596,False,True
34,GOOGL,2025-07-23,10-Q,596,True,True
35,GOOGL,2025-10-29,10-Q,596,True,True
65,TSLA,2024-04-23,10-Q,329,False,True
66,TSLA,2024-07-23,10-Q,329,True,True


## 7. Validation helpers
`compare_profiles` gives before vs after; `flag_summary` counts every flag by symbol. These feed `outputs/validation_report.md`.

In [17]:
raw = {"prices": prices_raw, "news": news_raw, "filings": filings_raw}
clean = {"prices": prices, "news": news, "filings": filings}
comp = cp.compare_profiles(raw, clean)
assert (comp["rows_before"] == comp["rows_after"]).all()
assert (comp["dup_id_after"] == 0).all() and (comp["dup_symbol_date_after"] == 0).all()
comp

,rows_before,missing_before,dup_id_before,dup_symbol_date_before,symbols_before,rows_after,missing_after,dup_id_after,dup_symbol_date_after,symbols_after
table,,,,,,,,,,
prices,2656,0,0,0,8,2656,8,0,0,8
news,3888,0,0,0,8,3888,482,0,0,8
filings,73,0,0,0,8,73,0,0,0,8


In [18]:
cp.flag_summary(prices, news, filings)

,flag_large_move,flag_placeholder,flag_lookahead,non_trading_news,flag_risk_repeat,flag_risk_reference_only
symbol,,,,,,
AAPL,1,0,0,154,0,2
ADBE,2,1,1,154,0,0
AMZN,0,1,0,154,3,0
GOOGL,0,3,0,154,2,3
META,1,2,0,154,0,0
MSFT,0,1,0,154,0,0
NVDA,2,0,0,154,0,0
TSLA,3,0,0,154,2,4
TOTAL,9,8,1,1232,7,9


## 8. Full pipeline
`run_pipeline` chains everything, saves `data/clean/*.parquet` and writes the report. Reading the saved files back should give the same tables we checked above.

In [19]:
out = cp.run_pipeline()
for name in ["prices", "news", "filings"]:
    saved = pd.read_parquet(f"data/clean/{name}_clean.parquet")
    assert saved.shape == clean[name].shape, name
    print(f"{name:8} saved {saved.shape}")
print("pipeline: all checks passed")

prices   saved (2656, 11)
news     saved (3888, 10)
filings  saved (73, 10)
pipeline: all checks passed
